# Spark SQL & DataFrame API — Beginner Training

**Topics Covered:**
- Create Temp Views with `.createOrReplaceTempView()`
- Run SQL queries using `spark.sql("SELECT ...")`
- SQL vs DataFrame API — when to use which
- Subqueries and CTEs in Spark SQL
- Mix SQL and DataFrame API in the same pipeline

> **Environment:** Databricks. Run cells top to bottom.

---
## 0. Setup — Create Sample DataFrames

We will use three small DataFrames throughout this notebook:
- **employees** — employee details
- **departments** — department master
- **salaries** — salary per employee

> Run this cell **first** before trying any other section.

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, avg, max, count, when, round

# ── employees ──────────────────────────────────────────────────────────────────
employees_data = [
    Row(emp_id=1, name="Ananya",  dept_id=10, city="Chennai",    join_year=2018),
    Row(emp_id=2, name="Ravi",    dept_id=20, city="Bangalore",  join_year=2019),
    Row(emp_id=3, name="Meena",   dept_id=10, city="Chennai",    join_year=2020),
    Row(emp_id=4, name="Karthik", dept_id=30, city="Hyderabad",  join_year=2017),
    Row(emp_id=5, name="Divya",   dept_id=20, city="Bangalore",  join_year=2021),
    Row(emp_id=6, name="Suresh",  dept_id=30, city="Hyderabad",  join_year=2016),
    Row(emp_id=7, name="Priya",   dept_id=10, city="Chennai",    join_year=2022),
    Row(emp_id=8, name="Arjun",   dept_id=20, city="Mumbai",     join_year=2015),
]

# ── departments ────────────────────────────────────────────────────────────────
departments_data = [
    Row(dept_id=10, dept_name="Engineering"),
    Row(dept_id=20, dept_name="Marketing"),
    Row(dept_id=30, dept_name="Finance"),
]

# ── salaries ───────────────────────────────────────────────────────────────────
salaries_data = [
    Row(emp_id=1, salary=75000),
    Row(emp_id=2, salary=82000),
    Row(emp_id=3, salary=68000),
    Row(emp_id=4, salary=91000),
    Row(emp_id=5, salary=77000),
    Row(emp_id=6, salary=95000),
    Row(emp_id=7, salary=62000),
    Row(emp_id=8, salary=88000),
]

emp_df    = spark.createDataFrame(employees_data)
dept_df   = spark.createDataFrame(departments_data)
salary_df = spark.createDataFrame(salaries_data)

print("Sample DataFrames created successfully!")
emp_df.show()


---
## 1. Create Temp Views with `.createOrReplaceTempView()`

A **Temp View** is a virtual table registered from a DataFrame.

- It lives only for the **current Spark session** — not persisted to disk.
- `createOrReplaceTempView` safely overwrites the view if it already exists (safe to re-run).
- Once registered, you can query it with standard SQL via `spark.sql()`.

**Syntax:**
```python
dataframe.createOrReplaceTempView("view_name")
```

In [0]:
# Register all three DataFrames as Temp Views
emp_df.createOrReplaceTempView("employees")
dept_df.createOrReplaceTempView("departments")
salary_df.createOrReplaceTempView("salaries")

print("Temp views registered: employees, departments, salaries")

# Verify using the Spark catalog
#display(spark.catalog.listTables())


In [0]:
# KEY POINT: Temp views are session-scoped.
# If the cluster restarts or a new session starts,
# you must re-run the registration cell above.

# Check if a specific view exists:
print(spark.catalog.tableExists("employees"))   # True
print(spark.catalog.tableExists("orders"))      # False — not registered


---
## 2. Run SQL Queries using `spark.sql()`

`spark.sql()` accepts a SQL string and **returns a DataFrame**.
You can call `.show()`, `display()`, or chain further transformations on the result.

**Syntax:**
```python
result_df = spark.sql("SELECT ... FROM view_name WHERE ...")
result_df.show()
```

In [0]:
# 2.1  Basic SELECT — all rows
tbl = ["employees","departments","salaries"]

for i in tbl:
    spark.sql(f"""SELECT * FROM {i}""").display()


In [0]:
# 2.2  Filter + ORDER BY
spark.sql("""
    SELECT name, city, join_year
    FROM   employees
    WHERE  city = 'Chennai'
    ORDER  BY join_year
""").show()


In [0]:
# 2.3  Aggregation — count employees per department
spark.sql("""
    SELECT dept_id,
           COUNT(*) AS emp_count
    FROM   employees
    GROUP  BY dept_id
    ORDER  BY emp_count DESC
""").show()


In [0]:
# 2.4  JOIN — combine employees with department names
spark.sql("""
    SELECT e.name,
           d.dept_name,
           e.city,
           e.join_year
    FROM   employees   e
    JOIN   departments d ON e.dept_id = d.dept_id
    ORDER  BY d.dept_name, e.name
""").show()


In [0]:
# Reading a table directly from the catalog
spark.sql("select * from employeeproject.default.employee").display()

---
## 3. SQL vs DataFrame API — When to Use Which?

Both produce **exactly the same result**.
Spark compiles both to the same physical execution plan via the Catalyst Optimizer.
The choice is purely about **readability and context**.

| Scenario | Prefer |
|---|---|
| Team has strong SQL background | SQL API |
| Simple aggregations / JOINs | Either |
| Complex programmatic logic (loops, conditions, Python variables) | DataFrame API |
| Reusable Python functions / unit testing | DataFrame API |
| Quick ad-hoc exploration in a notebook | SQL API |

Below is the **same query written both ways** — average salary per department.

In [0]:
# ── Approach A: SQL API ────────────────────────────────────────────────────────
print("=== SQL API ===")
spark.sql("""
    SELECT d.dept_name,
           ROUND(AVG(s.salary), 2) AS avg_salary
    FROM   employees   e
    JOIN   departments d ON e.dept_id = d.dept_id
    JOIN   salaries    s ON e.emp_id  = s.emp_id
    GROUP  BY d.dept_name
    ORDER  BY avg_salary DESC
""").show()


In [0]:
# ── Approach B: DataFrame API ──────────────────────────────────────────────────
print("=== DataFrame API ===")
(
    emp_df
    .join(dept_df,   emp_df.dept_id == dept_df.dept_id,  "inner")
    .join(salary_df, emp_df.emp_id  == salary_df.emp_id, "inner")
    .groupBy(dept_df.dept_name)
    .agg(round(avg(salary_df.salary), 2).alias("avg_salary"))
    .orderBy(col("avg_salary").desc())
    .show()
)


> **Trainer note:** Both outputs are identical.
> SQL reads naturally for analysts.
> DataFrame API is preferred when building reusable pipelines or when logic
> involves Python variables, dynamic column lists, or conditional branching.

---
## 4. Subqueries and CTEs in Spark SQL

Spark SQL supports both **subqueries** (inline queries) and **CTEs** (Common Table Expressions).

### 4a. Subquery in WHERE clause
Find employees who earn **above the overall average salary**.

In [0]:
spark.sql("""
    SELECT e.name,
           s.salary,
           e.city
    FROM   employees e
    JOIN   salaries  s ON e.emp_id = s.emp_id
    WHERE  s.salary > (SELECT AVG(salary) FROM salaries)
    ORDER  BY s.salary DESC
""").show()


### 4b. Scalar Subquery in SELECT list
Show each employee's salary alongside the overall average for easy comparison.

In [0]:
spark.sql("""
    SELECT e.name,
           s.salary,
           ROUND((SELECT AVG(salary) FROM salaries), 2)                  AS overall_avg,
           s.salary - ROUND((SELECT AVG(salary) FROM salaries), 2)       AS diff_from_avg
    FROM   employees e
    JOIN   salaries  s ON e.emp_id = s.emp_id
    ORDER  BY diff_from_avg DESC
""").show()


### 4c. CTE — WITH clause (single CTE)
Find the **highest-paid employee in each department**.

**Pattern:**
```sql
WITH cte_name AS (
    SELECT ...
)
SELECT * FROM cte_name
```

In [0]:
spark.sql("""
    WITH dept_max_salary AS (
        SELECT e.dept_id,
               MAX(s.salary) AS max_salary
        FROM   employees e
        JOIN   salaries  s ON e.emp_id = s.emp_id
        GROUP  BY e.dept_id
    )
    SELECT e.name,
           d.dept_name,
           s.salary
    FROM   employees      e
    JOIN   departments    d  ON e.dept_id  = d.dept_id
    JOIN   salaries       s  ON e.emp_id   = s.emp_id
    JOIN   dept_max_salary dm ON e.dept_id  = dm.dept_id
                              AND s.salary  = dm.max_salary
    ORDER  BY d.dept_name
""").show()


### 4d. Chained CTEs — multiple CTEs in one query
Label each employee as **Above Avg** or **Below Avg** relative to their department.

In [0]:
spark.sql("""
    WITH employee_salaries AS (
        -- Step 1: join employees + salaries
        SELECT e.emp_id,
               e.name,
               e.dept_id,
               e.join_year,
               s.salary
        FROM   employees e
        JOIN   salaries  s ON e.emp_id = s.emp_id
    ),
    dept_avg AS (
        -- Step 2: compute average salary per department
        SELECT dept_id,
               ROUND(AVG(salary), 2) AS avg_salary
        FROM   employee_salaries
        GROUP  BY dept_id
    )
    -- Step 3: label each employee vs their department average
    SELECT es.name,
           d.dept_name,
           es.salary,
           da.avg_salary        AS dept_avg,
           CASE
               WHEN es.salary >= da.avg_salary THEN 'Above Avg'
               ELSE                                 'Below Avg'
           END                  AS salary_band
    FROM   employee_salaries es
    JOIN   departments d  ON es.dept_id = d.dept_id
    JOIN   dept_avg    da ON es.dept_id = da.dept_id
    ORDER  BY d.dept_name, es.salary DESC
""").show()


> **Key takeaway:**
> - Use **subqueries** for simple one-off filters or scalar lookups.
> - Use **CTEs** for multi-step logic — they are easier to read, debug, and maintain.
> - Each CTE block can reference the ones defined before it.

---
## 5. Mix SQL and DataFrame API in the Same Pipeline

A real-world pipeline often cleans raw data, applies transformations, and produces
a final output. You can freely mix SQL and DataFrame API because `spark.sql()`
always returns a DataFrame.

**Typical pattern:**
```
Raw DataFrame
    → (DataFrame API)  clean / cast / rename
    → createOrReplaceTempView()
    → (spark.sql)      aggregate / join using SQL
    → (DataFrame API)  final column shaping / write
```

### Pipeline: Employee Salary Report
**Goal:** Produce a report showing each employee's name, department,
salary, department average, and a salary band label.

In [0]:
# ── STEP 1 (DataFrame API): Clean the raw data ────────────────────────────────
cleaned_emp_df = (
    emp_df
    .withColumnRenamed("name", "employee_name")
    .withColumn("join_year", col("join_year").cast("int"))
    .dropna()
)

cleaned_salary_df = (
    salary_df
    .withColumn("salary", col("salary").cast("double"))
    .filter(col("salary") > 0)          # remove invalid records
)

print("Step 1 complete — data cleaned using DataFrame API")
cleaned_emp_df.show()


In [0]:
# ── STEP 2 (DataFrame API → Temp View): Register cleaned data ─────────────────
cleaned_emp_df.createOrReplaceTempView("cleaned_employees")
cleaned_salary_df.createOrReplaceTempView("cleaned_salaries")

print("Step 2 complete — cleaned views registered")


In [0]:
# ── STEP 3 (SQL + CTE): Aggregate department-level averages ───────────────────
dept_summary_df = spark.sql("""
    WITH base AS (
        SELECT e.emp_id,
               e.employee_name,
               e.dept_id,
               e.join_year,
               s.salary
        FROM   cleaned_employees e
        JOIN   cleaned_salaries  s ON e.emp_id = s.emp_id
    ),
    dept_stats AS (
        SELECT dept_id,
               ROUND(AVG(salary), 2) AS dept_avg_salary,
               MAX(salary)           AS dept_max_salary
        FROM   base
        GROUP  BY dept_id
    )
    SELECT b.emp_id,
           b.employee_name,
           b.dept_id,
           b.join_year,
           b.salary,
           ds.dept_avg_salary,
           ds.dept_max_salary
    FROM   base       b
    JOIN   dept_stats ds ON b.dept_id = ds.dept_id
""")

print("Step 3 complete — department aggregation done via SQL + CTE")
dept_summary_df.show()


In [0]:
# ── STEP 4 (DataFrame API): Enrich + finalize the report ──────────────────────
final_report_df = (
    dept_summary_df
    .join(dept_df, dept_summary_df.dept_id == dept_df.dept_id, "left")
    .withColumn(
        "salary_band",
        when(col("salary") >= col("dept_avg_salary"), "Above Avg")
        .otherwise("Below Avg")
    )
    .select(
        col("employee_name").alias("Employee"),
        col("dept_name").alias("Department"),
        col("join_year").alias("Join Year"),
        col("salary").alias("Salary"),
        col("dept_avg_salary").alias("Dept Avg"),
        col("salary_band").alias("Salary Band")
    )
    .orderBy("Department", col("Salary").desc())
)

print("Step 4 complete — final report ready")
display(final_report_df)


### Pipeline Summary

| Step | API Used | What Happened |
|---|---|---|
| 1 | DataFrame API | Cleaned raw data — renamed columns, cast types, dropped nulls |
| 2 | DataFrame API | Registered cleaned DataFrames as Temp Views |
| 3 | SQL (CTE) | Joined and aggregated department-level averages using a multi-step CTE |
| 4 | DataFrame API | Added salary band label, selected and renamed final columns |

> **Best practice:** Use SQL for readable multi-table joins and aggregations.
> Use DataFrame API for programmatic logic — Python variables, conditional branching,
> dynamic column lists, or when writing reusable functions and unit tests.

---
## Quick Reference Card

| Operation | SQL API | DataFrame API |
|---|---|---|
| Register a view | — | `df.createOrReplaceTempView("name")` |
| Run a query | `spark.sql("SELECT ...")` | `df.select(...).filter(...)` |
| Filter rows | `WHERE condition` | `.filter(condition)` |
| Aggregation | `GROUP BY` in SQL string | `.groupBy().agg()` |
| Join | `JOIN ... ON` in SQL string | `.join(other_df, condition, how)` |
| Add a column | `SELECT *, expr AS col` | `.withColumn("col", expr)` |
| CTE | `WITH cte AS (...)` in SQL string | Intermediate DataFrame + new view |
| Subquery | Nested `SELECT` inside SQL | Intermediate DataFrame + join |
| Check view exists | — | `spark.catalog.tableExists("name")` |
| List registered views | — | `spark.catalog.listTables()` |